In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model in a `while True` loop and checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls in the response, it breaks out of the loop.

So the stop condition is: **no function calls this turn**.


In [3]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xfc78974a5240959565cf20dc7b651332",
        "span_id": "0xea963e58af0a9bd7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-26T20:11:34.483153Z",
    "end_time": "2026-07-26T20:11:34.483184Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "9a90054a-fb28-42fb-83e4-1738a0196ff3",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


## Q1. First trace

In [6]:
from starter import client, index  # or import RAGBase from rag_helper
from rag_helper import RAGBase
from opentelemetry import trace

tracer = trace.get_tracer("llm-zoomcamp")

class RAGTraced(RAGBase):
    def rag(self, query: str):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

    def search(self, query: str):
        with tracer.start_as_current_span("search") as span:
            return super().search(query)

    def llm(self, prompt: str):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            return response

In [7]:
# Instantiate your traced subclass instead of RAGBase directly
rag_traced = RAGTraced(index=index, llm_client=client)

# Call the rag method as you normally would
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x27a2d9a1d22172ad3c8a5729103e8c3e",
        "span_id": "0x6026a2bdc67b01ef",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x44a61ecf426ecd8f",
    "start_time": "2026-07-26T20:12:24.032938Z",
    "end_time": "2026-07-26T20:12:24.037965Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "9a90054a-fb28-42fb-83e4-1738a0196ff3",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x27a2d9a1d22172ad3c8a5729103e8c3e",
        "span_id": "0x16e1f2a0dfe0c586",
        "trace_state": "[]"
    },
    "kind": "SpanKind

### Answer: 3

## Q2. Capturing metrics as span attributes

### Answer: 7111 --> 7000

## Q3. Span timing

### Answer: 379 - 329 = 50 --> Under 100 ms